In [111]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

In [112]:
df_train_ria = pd.read_csv('/kaggle/input/datasets/samvelgalstyan/csv-files/Ria_pars.csv')
df_train_lenta = pd.read_csv('/kaggle/input/datasets/samvelgalstyan/csv-files/LentaRu_pars.csv')
df_test = pd.read_csv('/kaggle/input/datasets/samvelgalstyan/csv-files/test_news.csv')
df_sub = pd.read_csv('/kaggle/input/datasets/samvelgalstyan/csv-files/base_submission_news.csv')

In [113]:
df_train_lenta.shape[0]

642

In [114]:
def clean_text(text):
    text = re.sub(r'^.*?РИА Новости\.\s*', ' ', text)
    text = re.sub(r'полный текст интервью читайте на сайте ria ru', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'[0-9,\.\-\–\—\«\»\"\"\/\:\%]', ' ', text)
    text = re.sub(r'\b[а-яё]\b', ' ', text)
    text = re.sub(r'[a-zA-Z]', ' ', text)
    text = text.lower()
    return text

In [115]:
df_train = pd.concat([df_train_ria, df_train_lenta])

In [116]:
df_train = df_train.drop('Unnamed: 0',axis=1)

In [117]:
df_train['Text'] = df_train['Text'].apply(clean_text)
df_test['content'] = df_test['content'].apply(clean_text)

In [118]:
df_train = df_train.drop_duplicates(subset=['Text']).reset_index(drop=True)

In [119]:
from collections import Counter

In [120]:
for i in df_train['Category'].unique():
    text = ' '.join(df_train[df_train['Category'] == i]['Text']).lower()
    words = [j for j in text.split() if len(j) > 4]
    print(i, Counter(words).most_common(10))

economy [('также', 121), ('россии', 113), ('рублей', 113), ('будет', 92), ('можно', 91), ('которые', 90), ('более', 90), ('могут', 81), ('может', 78), ('процентов', 76)]
sport [('после', 88), ('играх', 85), ('олимпиаде', 79), ('олимпийских', 75), ('февраля', 70), ('место', 64), ('балла', 63), ('россии', 59), ('также', 53), ('италии', 49)]
tourism [('можно', 60), ('россии', 57), ('время', 53), ('также', 49), ('ранее', 48), ('после', 46), ('февраля', 46), ('туристов', 43), ('тысяч', 40), ('только', 35)]
science [('более', 113), ('когда', 98), ('после', 92), ('также', 91), ('может', 90), ('которые', 85), ('который', 83), ('время', 78), ('только', 78), ('ранее', 59)]
ussr [('россии', 96), ('украины', 95), ('которые', 67), ('после', 63), ('когда', 62), ('также', 55), ('только', 54), ('ранее', 52), ('украина', 52), ('словам', 49)]
forces [('после', 97), ('россии', 78), ('также', 62), ('который', 57), ('время', 57), ('когда', 52), ('рублей', 51), ('однако', 47), ('данным', 44), ('области', 41

In [121]:
barer = ['вновь','ноября','между','похоже','после','взятия','назад','января','года','хочет',
         'привлеч','были','новый','для','не','исполнили','сентября','Вооруженные силы (ВС) России',
         'начали','нанесли','ночь','на','по','октябре','об этом','в конце','августа','заявил','что',
         'из-за','едва','самой','рублей','также','процентов','можно','которые','могут','более','может',
         'компании','ранее','время','февраля','данным','который','когда','области','словам','этого',
         'однако','может','тысяч','италии','будет','только','стало','именно','чтобы','несколько','человек',
         'предупредила','назвала','сообщалось','тысячи','очень','через','рублей','здесь','сообщает',
         'ленте','всего','говорит','будут','например','которых','рублей)','кроме','iphone','километров',
         'которая','затем','рассказала','лудей','например','секса','посоветовала','добавила','больше','случае',
         'петросян','произвольный','уточняется','известно','стоит','почти','которой','россии','ворошилов','первый','жизни',
         'жизнь','ракеты','тогда','часть','сейчас','стала','произвольной','короткой','попробовать','около','лучше','людей',
         'ильюшин', 'пирогов','сталин','работы','ранеесообщалось','получил','жигадло','сразу','вместе','спустя','момент',
         'кэтролл','человека','хопкинс','актер','людей','много','часто','годар','времени','первой','улице','мужчина','среди',
         'сообщили','самых','сегодня','одной','одного','стали'
        ]

In [122]:
df_train['Text'] = df_train['Text'].apply(lambda x: ' '.join([w for w in str(x).split() if w.lower() not in barer]))
df_test['content'] = df_test['content'].apply(lambda x: ' '.join([w for w in str(x).split() if w.lower() not in barer]))

In [123]:
df_train = df_train[df_train['Category'] != 'wellness']

In [124]:
df_train = df_train[df_train['Category'] != 'science']

In [125]:
df_train.loc[df_train['Category'] == 'economy', 'Category'] = 1

In [126]:
df_train.loc[df_train['Category'] == 'forces', 'Category'] = 2

In [127]:
df_train.loc[df_train['Category'] == 'ussr', 'Category'] = 3

In [128]:
df_train.loc[df_train['Category'] == 'sport', 'Category'] = 4

In [129]:
df_train.loc[df_train['Category'] == 'tourism', 'Category'] = 7

In [130]:
df_train['Category'] = df_train['Category'].astype(int)

In [131]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 480 entries, 0 to 591
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  480 non-null    int64 
 1   Text      480 non-null    object
dtypes: int64(1), object(1)
memory usage: 11.2+ KB


In [132]:
df_train = df_train.sample(frac=1).reset_index(drop=True)

In [133]:
X_train = df_train['Text']
y_train = df_train['Category']
X_test = df_test['content'].copy()

In [134]:
X_test

0        фото фонтанка ру поделитьсяэкс министру оборон...
1        в начале пушкинском районе санкт петербурга вв...
2        фото анастасия борисова международная федераци...
3        если вы хотели но так съездили море летом чита...
4        сергей пиняев фото алексей филиппов риа новост...
                               ...                        
26270    фото риа новости алевтина запольская главное у...
26271    вадим гутцайт фото алевтина запольская министр...
26272    фото олег харсеев коммерсантъ александр курбат...
26273    владимир зеленский фото варвара кошечкина през...
26274    фото алевтина запольская азербайджану нужна но...
Name: content, Length: 26275, dtype: object

In [135]:
from sklearn.model_selection import train_test_split

In [136]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [137]:
import torch
from transformers import AutoTokenizer, AutoModel

In [138]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [139]:
from transformers import AutoTokenizer, AutoModel
import torch
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [140]:
def get_bert_embeddings(texts, batch_size=64):
    model.eval() 
    all_embeddings = []
    
    with torch.no_grad(): 
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(device)
            
            outputs = model(**tokens) 
            
            embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)


In [141]:
X_train_vec = get_bert_embeddings(X_train.tolist())
X_val_vec = get_bert_embeddings(X_val.tolist())
X_test = get_bert_embeddings(X_test.tolist())

In [142]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [143]:
clf = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced')
clf.fit(X_train_vec, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [144]:
y_pred = clf.predict(X_val_vec)

In [145]:
accuracy_score(y_val, y_pred)

0.8854166666666666

In [146]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           1       0.79      0.90      0.84        21
           2       0.94      0.79      0.86        19
           3       0.76      1.00      0.87        13
           4       1.00      1.00      1.00        22
           7       0.94      0.76      0.84        21

    accuracy                           0.89        96
   macro avg       0.89      0.89      0.88        96
weighted avg       0.90      0.89      0.89        96



In [147]:
y_pred = clf.predict(X_test)

### for Kaggle

In [148]:
df_sub['topic'] = y_pred

In [149]:
df_sub.to_csv('submission.csv', index=False, index_label=False)

### for saving model

In [ ]:
import joblib

In [ ]:
joblib.dump(clf, 'news_classifier_model.pkl')